# Day 2-3 Lab, Part 1: scikit-learn Style ML Basics

**NIAI AI/ML Program · Module 3-4 · Trainer: Anique Atique Alam**

**How this notebook works**
- Cells marked `# TODO` are yours to complete. Cells with `assert` check your work.
- Sections show which lecture part they need: **II** (ML foundations), **III** (linear regression), **IV** (logistic regression).
- All datasets are bundled with scikit-learn, so no network is needed.
- Kernel: `niai-ml` (select it in the top right of JupyterLab).

| Section | Topic | Needs | Time |
|---|---|---|---|
| 1 | The estimator API | II | 15 min |
| 2 | Metrics, baselines, cross-validation | II | 15 min |
| 3 | Pipelines and leakage | II | 10 min |
| 4 | Build your own estimator (gradient descent) | III | 25 min |
| 5 | Polynomial regression and Ridge | III | 15 min |
| 6 | Logistic regression | IV | 20 min |
| 7 | Checkpoint questions | | 5 min |

In [ ]:
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.base import BaseEstimator, RegressorMixin
from sklearn.compose import ColumnTransformer
from sklearn.datasets import load_breast_cancer, load_diabetes, load_iris
from sklearn.dummy import DummyRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression, LogisticRegression, Ridge
from sklearn.metrics import (confusion_matrix, f1_score, mean_absolute_error,
                             mean_squared_error, precision_score, recall_score,
                             roc_auc_score)
from sklearn.model_selection import GridSearchCV, cross_val_score, train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, PolynomialFeatures, StandardScaler

RANDOM_STATE = 42
plt.rcParams["figure.figsize"] = (6, 3.5)

## 1. The estimator API (Part II)

Every scikit-learn model follows the same contract:

| Method | Meaning |
|---|---|
| `fit(X, y)` | learn from the training data |
| `predict(X)` | produce predictions |
| `score(X, y)` | default quality metric (R² for regressors, accuracy for classifiers) |
| `transform(X)` | (preprocessors) convert data using what `fit` learned |

Learned values end with an underscore: `coef_`, `intercept_`.

We start with the diabetes dataset: 10 numeric features, predict disease progression.

In [ ]:
X, y = load_diabetes(return_X_y=True, as_frame=True)
print(X.shape, y.shape)
X.head()

In [ ]:
# TODO: split into 80% train / 20% test, using random_state=RANDOM_STATE
X_train, X_test, y_train, y_test = ...

In [ ]:
# TODO 1: create a LinearRegression model
model = ...

# TODO 2: train it on the training data
...

# TODO 3: predict on the test data
y_pred = ...

In [ ]:
assert X_train.shape[0] == 353
assert model.coef_.shape == (10,)
assert y_pred.shape == y_test.shape
print("R^2 on test:", round(model.score(X_test, y_test), 3))
pd.Series(model.coef_, index=X.columns).sort_values()

## 2. Metrics, baselines and cross-validation (Part II)

A score means nothing without a **baseline**. First compute the three regression metrics yourself, then compare to a model that always predicts the training mean.

In [ ]:
# TODO: compute by hand with numpy (no sklearn.metrics)
#   MAE  = mean(|error|)      RMSE = sqrt(mean(error^2))
#   R2   = 1 - sum(error^2) / sum((y - mean(y))^2)
err = y_test - y_pred
mae = ...
rmse = ...
r2 = ...

In [ ]:
assert np.isclose(mae, mean_absolute_error(y_test, y_pred))
assert np.isclose(rmse, np.sqrt(mean_squared_error(y_test, y_pred)))
assert np.isclose(r2, model.score(X_test, y_test))
print(f"MAE {mae:.1f}   RMSE {rmse:.1f}   R2 {r2:.3f}")

In [ ]:
# TODO: baseline = DummyRegressor(strategy="mean"), fit on the training data
baseline = ...
...
base_r2 = baseline.score(X_test, y_test)

In [ ]:
assert base_r2 < 0.05, "a mean predictor should have R^2 near 0"
assert model.score(X_test, y_test) > base_r2
print(f"baseline R2 {base_r2:.3f}  vs  model R2 {model.score(X_test, y_test):.3f}")

One split can be lucky or unlucky. **Cross-validation** averages over several splits.

In [ ]:
# TODO: 5-fold cross-validated R^2 for a fresh LinearRegression on the FULL X, y
scores = ...

In [ ]:
assert len(scores) == 5
print("fold scores:", scores.round(3))
print(f"mean {scores.mean():.3f} +/- {scores.std():.3f}")

## 3. Pipelines and leakage (Part II)

Scalers and imputers **learn** statistics. If they learn them from the whole dataset, test information leaks into training. A `Pipeline` keeps preprocessing and model together so it is refit correctly each time.

In [ ]:
Xc, yc = load_breast_cancer(return_X_y=True, as_frame=True)
Xc_tr, Xc_te, yc_tr, yc_te = train_test_split(
    Xc, yc, test_size=0.2, stratify=yc, random_state=RANDOM_STATE)

scaler_all = StandardScaler().fit(Xc)      # WRONG: has seen the test rows
# TODO: the RIGHT way: fit a scaler on the training rows only
scaler_train = ...

In [ ]:
leak_gap = np.abs(scaler_all.mean_ - scaler_train.mean_).max()
assert leak_gap > 0
print("largest difference in a learned mean:", round(leak_gap, 3))

In [ ]:
# TODO: build a pipeline: StandardScaler then LogisticRegression(max_iter=1000)
pipe = ...
pipe.fit(Xc_tr, yc_tr)

In [ ]:
assert isinstance(pipe.steps[0][1], StandardScaler)
assert pipe.score(Xc_te, yc_te) > 0.95
print("test accuracy:", round(pipe.score(Xc_te, yc_te), 3))

Real tables mix numeric and categorical columns, with missing values. `ColumnTransformer` applies the right steps to the right columns, which is exactly what your `quakes` project does.

In [ ]:
df_mixed = pd.DataFrame({
    "age":    [25, 32, np.nan, 41, 29, 55],
    "income": [40, 52, 61, np.nan, 45, 90],
    "city":   ["Lahore", "Karachi", "Lahore", np.nan, "Quetta", "Karachi"],
})

# TODO: numeric (age, income): median imputation, then StandardScaler
#       nominal (city): most-frequent imputation, then OneHotEncoder(handle_unknown="ignore", sparse_output=False)
numeric = ...
nominal = ...
prep = ColumnTransformer([...])
Xt = prep.fit_transform(df_mixed)

In [ ]:
assert Xt.shape == (6, 5), f"expected 2 numeric + 3 city columns, got {Xt.shape}"
assert not np.isnan(Xt).any()
print(prep.get_feature_names_out())

## 4. Build your own estimator: gradient descent (Part III)

Model: $\hat y = \mathbf{x}^\top \mathbf{w} + b$. Loss: $J = \frac{1}{2m}\sum_i (\hat y_i - y_i)^2$.

Gradients: $\frac{\partial J}{\partial w_j} = \frac{1}{m}\sum_i (\hat y_i - y_i)\,x_{ij}$ and $\frac{\partial J}{\partial b} = \frac{1}{m}\sum_i (\hat y_i - y_i)$.

Update: $w_j \leftarrow w_j - \alpha \,\partial J / \partial w_j$.

You will write the gradient twice (loops, then vectorized), check they agree, time them, and wrap the vectorized one in a scikit-learn style class.

In [ ]:
rng = np.random.default_rng(RANDOM_STATE)
X_syn = rng.normal(size=(500, 3))
w_true, b_true = np.array([2.0, -1.0, 0.5]), 3.0
y_syn = X_syn @ w_true + b_true + rng.normal(0, 0.5, 500)

In [ ]:
def gradient_loop(X, y, w, b):
    """Gradient of J using plain Python loops. Returns (grad_w, grad_b)."""
    m, n = X.shape
    grad_w = np.zeros(n)
    grad_b = 0.0
    for i in range(m):
        # TODO: prediction for sample i, using a loop over features j
        pred = ...
        err = pred - y[i]
        # TODO: add this sample's contribution to grad_w[j] (every j) and to grad_b
        ...
    return grad_w / m, grad_b / m

In [ ]:
def gradient_vec(X, y, w, b):
    """Same gradient with NO Python loops. Returns (grad_w, grad_b)."""
    # TODO: err = X @ w + b - y ; grad_w = X.T @ err / m ; grad_b = err.mean()
    ...

In [ ]:
w0, b0 = rng.normal(size=3), 1.0
gw_l, gb_l = gradient_loop(X_syn, y_syn, w0, b0)
gw_v, gb_v = gradient_vec(X_syn, y_syn, w0, b0)
assert np.allclose(gw_l, gw_v) and np.isclose(gb_l, gb_v), "loop and vectorized gradients differ"
print("gradients agree")

In [ ]:
X_big, y_big = rng.normal(size=(20000, 5)), rng.normal(size=20000)
w5 = np.zeros(5)

t0 = time.perf_counter(); gradient_loop(X_big, y_big, w5, 0.0); t_loop = time.perf_counter() - t0
t0 = time.perf_counter(); gradient_vec(X_big, y_big, w5, 0.0);  t_vec = time.perf_counter() - t0
print(f"loop {t_loop*1000:.1f} ms   vectorized {t_vec*1000:.2f} ms   speedup x{t_loop/t_vec:.0f}")
assert t_vec < t_loop

In [ ]:
class GDLinearRegression(BaseEstimator, RegressorMixin):
    def __init__(self, lr=0.1, n_iter=1000):
        self.lr = lr
        self.n_iter = n_iter

    def fit(self, X, y):
        X = np.asarray(X, dtype=float)
        y = np.asarray(y, dtype=float)
        w, b = np.zeros(X.shape[1]), 0.0
        self.loss_history_ = []
        for _ in range(self.n_iter):
            # TODO 1: append the current loss (1/2m) * sum(err**2) to loss_history_
            # TODO 2: get gradients from gradient_vec, then update w and b
            ...
        self.coef_, self.intercept_ = w, b
        return self

    def predict(self, X):
        # TODO: X @ coef_ + intercept_
        ...

In [ ]:
gd = GDLinearRegression(lr=0.1, n_iter=1000).fit(X_syn, y_syn)
ref = LinearRegression().fit(X_syn, y_syn)
assert gd.loss_history_[-1] < gd.loss_history_[0]
assert np.allclose(gd.coef_, ref.coef_, atol=0.02), "should match the closed-form solution"
print("GD coef:", gd.coef_.round(3), " sklearn coef:", ref.coef_.round(3))

**Learning rate.** Fit with three learning rates and watch the loss curve. One is too small, one is right, one is too large.

In [ ]:
learning_rates = [0.001, 0.1, 2.5]
histories = {}
for lr in learning_rates:
    # TODO: fit GDLinearRegression(lr=lr, n_iter=60) on X_syn, y_syn and store loss_history_ in histories[lr]
    ...

In [ ]:
for lr, h in histories.items():
    plt.plot(h, label=f"lr={lr}")
plt.yscale("log"); plt.xlabel("iteration"); plt.ylabel("loss"); plt.legend(); plt.show()
assert histories[0.1][-1] < histories[0.001][-1]
assert histories[2.5][-1] > histories[2.5][0], "lr=2.5 should diverge"

**Feature scaling.** The same algorithm on features with very different scales:

In [ ]:
scales = np.array([1.0, 50.0, 0.1])
X_raw = X_syn * scales
y_raw = X_raw @ np.array([2.0, -0.04, 10.0]) + 3.0 + rng.normal(0, 0.5, 500)
X_scaled = StandardScaler().fit_transform(X_raw)

# TODO: fit on the UNSCALED data with lr=1e-4, n_iter=500 (a larger lr would blow up the big feature)
#       and on the SCALED data with lr=0.1, n_iter=500. Keep the fitted models.
gd_raw = ...
gd_scaled = ...

In [ ]:
print(f"final loss unscaled {gd_raw.loss_history_[-1]:.3f}   scaled {gd_scaled.loss_history_[-1]:.3f}")
assert gd_scaled.loss_history_[-1] < gd_raw.loss_history_[-1] / 10

## 5. Polynomial regression and Ridge (Part III)

A polynomial model is still *linear in the weights*: expand the features ($x, x^2, x^3, \dots$), then fit a linear model. The degree controls flexibility.

In [ ]:
rng = np.random.default_rng(1)
x = np.sort(rng.uniform(0, 2 * np.pi, 60))
y_sin = np.sin(x) + rng.normal(0, 0.3, 60)
Xp = x.reshape(-1, 1)
Xp_tr, Xp_val, yp_tr, yp_val = train_test_split(Xp, y_sin, test_size=0.3, random_state=RANDOM_STATE)

In [ ]:
degrees = range(1, 13)
train_err, val_err = [], []
for d in degrees:
    # TODO: pipeline PolynomialFeatures(d, include_bias=False) -> StandardScaler -> LinearRegression
    model_d = ...
    model_d.fit(Xp_tr, yp_tr)
    train_err.append(mean_squared_error(yp_tr, model_d.predict(Xp_tr)))
    val_err.append(mean_squared_error(yp_val, model_d.predict(Xp_val)))
best_degree = degrees[int(np.argmin(val_err))]

In [ ]:
plt.plot(list(degrees), train_err, "o-", label="train")
plt.plot(list(degrees), val_err, "o-", label="validation")
plt.yscale("log"); plt.xlabel("degree"); plt.ylabel("MSE"); plt.legend(); plt.show()
assert train_err[-1] < train_err[0]
assert 2 <= best_degree <= 9
print("best degree on validation:", best_degree)

**Regularisation.** At degree 12 the plain model chases noise with huge weights. Ridge penalises large weights.

In [ ]:
def poly_model(d, est):
    return make_pipeline(PolynomialFeatures(d, include_bias=False), StandardScaler(), est)

plain = poly_model(12, LinearRegression()).fit(Xp_tr, yp_tr)
# TODO: ridge = the same degree-12 model with Ridge(alpha=1.0), fit on the same data
ridge = ...

In [ ]:
norm_plain = np.linalg.norm(plain[-1].coef_)
norm_ridge = np.linalg.norm(ridge[-1].coef_)
print(f"weight norm  plain {norm_plain:.1f}   ridge {norm_ridge:.1f}")
print(f"validation MSE plain {mean_squared_error(yp_val, plain.predict(Xp_val)):.3f}   "
      f"ridge {mean_squared_error(yp_val, ridge.predict(Xp_val)):.3f}")
assert norm_ridge < norm_plain

**Stretch:** Ridge with `alpha=1.0` may over-smooth. Try `alpha` in `[0.001, 0.01, 0.1, 1]` and report which one gives the lowest validation MSE. Regularisation strength is a hyperparameter you tune, not a constant.

## 6. Logistic regression (Part IV)

Breast cancer data. We treat **malignant as the positive class** (1), because missing a malignant case is the costly error.

In [ ]:
yb = (yc == 0).astype(int)          # 1 = malignant
Xb_tr, Xb_te, yb_tr, yb_te = train_test_split(
    Xc, yb, test_size=0.2, stratify=yb, random_state=RANDOM_STATE)

# TODO: pipeline StandardScaler -> LogisticRegression(max_iter=1000); fit on the training data
clf = ...
...
# TODO: probability of the positive class for the test rows
proba = ...

In [ ]:
assert proba.shape == (len(Xb_te),)
auc = roc_auc_score(yb_te, proba)
assert auc > 0.95
print("ROC-AUC:", round(auc, 4))

In [ ]:
pred = (proba >= 0.5).astype(int)
# TODO: confusion matrix, precision, recall and F1 for `pred` (use sklearn.metrics)
cm = ...
precision = ...
recall = ...
f1 = ...

In [ ]:
assert cm.shape == (2, 2)
print(cm)
print(f"precision {precision:.3f}   recall {recall:.3f}   F1 {f1:.3f}")

**Threshold sweep.** 0.5 is a default, not a law. Lower thresholds catch more malignant cases and raise false alarms.

In [ ]:
rows = []
for t in np.arange(0.1, 0.95, 0.1):
    # TODO: predictions at threshold t, then append {"threshold", "precision", "recall"} to rows
    ...
table = pd.DataFrame(rows).round(3)
table

In [ ]:
# TODO: the HIGHEST threshold whose recall is still >= 0.95
chosen_threshold = ...

In [ ]:
p = (proba >= chosen_threshold).astype(int)
assert recall_score(yb_te, p) >= 0.95
print("chosen threshold:", chosen_threshold)

**Multiclass.** Iris has three classes. `LogisticRegression` handles them directly (softmax).

In [ ]:
Xi, yi = load_iris(return_X_y=True)
Xi_tr, Xi_te, yi_tr, yi_te = train_test_split(Xi, yi, test_size=0.3, stratify=yi, random_state=RANDOM_STATE)

# TODO: pipeline StandardScaler -> LogisticRegression(max_iter=1000), fit, then predict on the test set
iris_clf = ...
...
iris_pred = ...
iris_cm = confusion_matrix(yi_te, iris_pred)

In [ ]:
assert iris_cm.shape == (3, 3)
assert (iris_pred == yi_te).mean() > 0.85
probs = iris_clf.predict_proba(Xi_te[:3])
assert np.allclose(probs.sum(axis=1), 1)
print(iris_cm); print(probs.round(3))

**Tuning.** `C` is the *inverse* regularisation strength: small `C` means strong regularisation. Search over it with cross-validation. Note the pipeline keeps the scaler inside each fold.

In [ ]:
param_grid = {"logisticregression__C": [0.01, 0.1, 1, 10, 100]}
# TODO: GridSearchCV over a StandardScaler -> LogisticRegression pipeline with 5 folds, scoring="roc_auc"; fit on Xb_tr, yb_tr
search = ...
...

In [ ]:
assert search.best_score_ > 0.95
print(search.best_params_, round(search.best_score_, 4))

## 7. Checkpoint questions

Answer in the cells below (2-3 sentences each).

1. In section 3, what exactly leaked when the scaler was fit on all rows, and why does that make a score untrustworthy?
2. In section 4, why does the loop version lose to the vectorized one, and what happened to the loss with `lr=2.5`?
3. In section 5, what happened to training and validation error as the degree grew, and what did Ridge change?

**Your answer to Q1:**

_(write here)_

**Your answer to Q2:**

_(write here)_

**Your answer to Q3:**

_(write here)_